In [1]:
import open3d as o3d
import numpy as np
import copy

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


1. mesh preprocessing -> sampling, dowm sampling

In [2]:
def mesh_preprocess(mesh_e_path, 
                    mesh_r_path, 
                    mesh_sample_point = 10000000, 
                    down_sample_res = 0.05):
    mesh_e = o3d.io.read_triangle_mesh(mesh_e_path)
    mesh_r = o3d.io.read_triangle_mesh(mesh_r_path)

    eva_bbx = mesh_e.get_axis_aligned_bounding_box()
    min_bound = eva_bbx.get_min_bound()
    min_bound[2]-=down_sample_res
    max_bound = eva_bbx.get_max_bound()
    max_bound[2]+=down_sample_res
    eva_bbx = o3d.geometry.AxisAlignedBoundingBox(min_bound, max_bound) 
    mesh_r = mesh_r.crop(eva_bbx)

    pcd_e = mesh_e.sample_points_uniformly(number_of_points = mesh_sample_point)
    pcd_r = mesh_r.sample_points_uniformly(number_of_points = mesh_sample_point)

    if down_sample_res > 0:
        pcd_e_before = len(pcd_e.points)
        pcd_e_dowm = pcd_e.voxel_down_sample(down_sample_res)
        pcd_r_down = pcd_r.voxel_down_sample(down_sample_res)
        pcd_e_after = len(pcd_e_dowm.points)

        print("Predicted mesh unifrom sample: ", pcd_e_before, " --> ", pcd_e_after, " (", down_sample_res, "m)")

    return pcd_e_dowm, pcd_r_down

2. ICP -> because here it's putting the eyes on mappingn quality so if keep the error from z-axis it will led to the inaccuracy of the result

In [3]:
def icp_align_eval_to_ref(
    eva_pts, ref_pts, max_correspondence: float = 0.2,
):
    Tgt2_3x4 = np.array([
    [0.998349155766, 0.000474285562, -0.057434643153, 0.000521513371],
    [0.000317738837, 0.999904999508,  0.013780094406, 0.004363416389],
    [0.057435722534,-0.013775594833,  0.998254161406, 0.006349134107],
    ], dtype=np.float64)

    Tpr2_3x4 = np.array([
        [0.99829958, 0.00119610, -0.05827960, -0.00139293],
        [-0.00028225, 0.99987693, 0.01568610,  0.00027624],
        [0.05829119, -0.01564298, 0.99817706, -0.00050557],
    ], dtype=np.float64)

    def to4(T3x4):
        T = np.eye(4, dtype=np.float64)
        T[:3,:4] = T3x4
        return T

    T_gt2   = to4(Tgt2_3x4)
    T_pred2 = to4(Tpr2_3x4)

    T_init = T_gt2 @ np.linalg.inv(T_pred2)  # Pred -> GT
    
    reg = o3d.pipelines.registration.registration_icp(
        eva_pts,
        ref_pts,
        max_correspondence_distance = max_correspondence,
        init = T_init,
        estimation_method = o3d.pipelines.registration.TransformationEstimationPointToPoint(),
        criteria=o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=100)
    )

    print(reg.fitness, reg.inlier_rmse)
    print(reg.transformation)

    return reg.transformation.astype(np.float64)


def apply_transform(pcd, T):
    pts = np.asarray(pcd.points)  # (N,3)
    xyz1 = np.hstack([pts, np.ones((pts.shape[0], 1), dtype=pts.dtype)])
    out = (xyz1 @ T.T)[:, :3]

    pcd_new = o3d.geometry.PointCloud()
    pcd_new.points = o3d.utility.Vector3dVector(out)
    return pcd_new

3. NN distance

In [4]:
def nn_correspondence(verts1, verts2, truncation_dist, ignore_outlier=True):
    
    indices = []
    distances = []
    if len(verts1) == 0 or len(verts2) == 0:
        return indices, distances

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(verts1.astype(np.float64))
    kdtree = o3d.geometry.KDTreeFlann(pcd)

    truncation_dist_square = truncation_dist**2

    for vert in verts2:
        _, inds, dist_square = kdtree.search_knn_vector_3d(vert, 1)
        
        if dist_square[0] < truncation_dist_square:
            indices.append(inds[0])
            distances.append(np.sqrt(dist_square[0]))
        else:
            if not ignore_outlier:
                indices.append(inds[0])
                distances.append(truncation_dist)

    return indices, distances

4. evaluation for mesh

In [5]:
import numpy as np
import open3d as o3d
import copy

# 低饱和度颜色（RGB 0~1）
TEAL   = np.array([0.165, 0.616, 0.561], dtype=np.float64)  # #2A9D8F
MAROON = np.array([0.478, 0.180, 0.180], dtype=np.float64)  # #7A2E2E
LIGHTCORAL = np.array([240/255, 128/255, 128/255], dtype=np.float64)
INDIANRED = np.array([205/255, 92/255, 92/255], dtype=np.float64)  # #CD5C5C


def _slice_pcd(pcd: o3d.geometry.PointCloud, axis: int, width: float):

    pts = np.asarray(pcd.points)
    if pts.shape[0] == 0:
        return pcd
    c = 0.5 * (pts[:, axis].min() + pts[:, axis].max())
    m = np.abs(pts[:, axis] - c) <= 0.5 * width

    out = o3d.geometry.PointCloud()
    out.points = o3d.utility.Vector3dVector(pts[m])
    if pcd.has_colors():
        out.colors = o3d.utility.Vector3dVector(np.asarray(pcd.colors)[m])
    if pcd.has_normals():
        out.normals = o3d.utility.Vector3dVector(np.asarray(pcd.normals)[m])
    return out

def _draw_pretty(pcd_eval, pcd_gt, voxel,
                 point_size=3.0,
                 slice_axis=1, slice_width=2.0,
                 side_front=(1.0, 0.0, 0.15), up=(0.0, 0.0, 1.0),
                 bg=(0.97, 0.97, 0.97)):

    eva = copy.deepcopy(pcd_eval)
    gt  = copy.deepcopy(pcd_gt)

    eva.paint_uniform_color(MAROON)
    gt.paint_uniform_color(INDIANRED)

    eva = eva.voxel_down_sample(0.95)
    gt = gt.voxel_down_sample(0.95)

    # gt  = _slice_pcd(gt,  axis=slice_axis, width=slice_width)
    # eva = _slice_pcd(eva, axis=slice_axis, width=slice_width)

    obb = gt.get_oriented_bounding_box()
    GOLDENROD = np.array([218/255, 165/255, 32/255], dtype=np.float64)
    obb.color = tuple(GOLDENROD.tolist())

    # 用 Visualizer 控制点大小、背景、相机
    vis = o3d.visualization.Visualizer()
    vis.create_window(width=1600, height=900, visible=True)
    vis.add_geometry(gt)
    vis.add_geometry(eva)
    vis.add_geometry(obb)

    opt = vis.get_render_option()
    opt.background_color = np.asarray(bg, dtype=np.float64)
    opt.point_size = float(point_size)
    opt.light_on = True

    # 锁定“侧视但略带俯仰”的视角，避免完全侧视压扁
    bbox = gt.get_axis_aligned_bounding_box()
    lookat = bbox.get_center().tolist()
    front  = np.asarray(side_front, dtype=np.float64)
    front  = (front / (np.linalg.norm(front) + 1e-12)).tolist()

    ctr = vis.get_view_control()
    ctr.set_lookat(lookat)
    ctr.set_front(front)
    ctr.set_up(list(up))
    ctr.set_zoom(0.8)

    vis.run()
    vis.destroy_window()



In [6]:
def eva_mesh(file_pred, file_gt, down_sample_res = 0.05, tau = 0.2, truncation_acc=1, truncation_com=1, vis = True):
    
    #down sampling
    pcd_e_down, pcd_r_down = mesh_preprocess(file_pred, 
                                             file_gt, 
                                             down_sample_res = 0.05, 
                                             mesh_sample_point=100000)
    
    if vis:
        _draw_pretty(pcd_e_down, pcd_r_down,
                    voxel=down_sample_res,
                    point_size=3.5,
                    slice_axis=1,        # 0:x  1:y  2:z，按你的长轴/侧视方向改
                    slice_width=40.0,     # 切片厚度（米），越小越“透”
                    side_front=(1,0,0.18))
    
    #icp
    T_align = np.eye(4, dtype=np.float64)
    pcd_e_down_copy = copy.deepcopy(pcd_e_down)
    pcd_r_down_copy = copy.deepcopy(pcd_r_down)
    T_align = icp_align_eval_to_ref(pcd_e_down_copy, pcd_r_down_copy, max_correspondence=1.0)
    pcd_e_down = apply_transform(pcd_e_down, T_align)
    
    # if vis:
    #     r = max(3.0 * 0.5, 0.05)
    #     pcd_e_down.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=r, max_nn=40))
    #     pcd_e_down.normalize_normals()
    #     _draw_pretty(pcd_e_down, pcd_r_down,
    #                 voxel=down_sample_res,
    #                 point_size=3.5,
    #                 slice_axis=1,        # 0:x  1:y  2:z，按你的长轴/侧视方向改
    #                 slice_width=40.0,     # 切片厚度（米），越小越“透”
    #                 side_front=(1,0,0.18))

    verts_e = np.asarray(pcd_e_down.points)
    verts_r = np.asarray(pcd_r_down.points)

    _, dist_e = nn_correspondence(verts_r, verts_e, truncation_acc, True) # find nn in ground truth samples for each predict sample -> precision related
    _, dist_r = nn_correspondence(verts_e, verts_r, truncation_com, False) # find nn in predict samples for each ground truth sample -> recall related

    dist_e = np.array(dist_e)
    dist_r = np.array(dist_r)

    dist_e_mean = np.mean(dist_e)
    dist_r_mean = np.mean(dist_r)

    chamfer_l1 = 0.5 * (dist_e_mean + dist_r_mean)
    precision = np.mean((dist_e < tau).astype('float')) * 100
    recall = np.mean((dist_r < tau).astype('float')) * 100.0
    fscore = 2 * precision * recall / (precision + recall)

    metrics = {'MAE_accuracy (m)': dist_e_mean,
               'MAE_completeness (m)': dist_r_mean,
               'Chamfer_L1 (m)': chamfer_l1, 
               'Precision [Accuracy] (%)': precision, 
               'Recall [Completeness] (%)': recall,
               'F-score (%)': fscore, 
               'Spacing (m)': down_sample_res,  # evlaution setup
               'Inlier_threshold (m)': tau,  # evlaution setup
               'Outlier_truncation_acc (m)': truncation_acc, # evlaution setup
               'Outlier_truncation_com (m)': truncation_com  # evlaution setup
               }

    return metrics

In [7]:
def main():

    eva_mesh_path = r"D:\project2\data\round1\mesh_result\gt.ply"
    eva_mesh_path_with_ekf = r"D:\project2\data\round1\mesh_result\pred_mesh_ekf.ply"
    ref_mesh_path = r"D:\project2\data\round1\mesh_result\pred_mesh.ply"
    kiss_mesh_path = r"D:\project2\data\round1\mesh_result\kiss_icp_mesh.ply"

    # res_pred = eva_mesh(eva_mesh_path, ref_mesh_path, down_sample_res = 0.05, tau = 0.2, truncation_acc=1, truncation_com=1, vis = True)
    # print(res_pred)
    
    # res_kiss = eva_mesh(kiss_mesh_path, ref_mesh_path, down_sample_res = 0.05, tau = 0.2, truncation_acc=1, truncation_com=1, vis = True)
    # print(res_kiss)

    res_ekf = eva_mesh(eva_mesh_path_with_ekf, ref_mesh_path, down_sample_res = 0.05, tau = 0.2, truncation_acc=1, truncation_com=1, vis = True)
    print(res_ekf)



if __name__ == "__main__":
    main()

Predicted mesh unifrom sample:  100000  -->  99775  ( 0.05 m)
0.991891756452017 0.42964615526840727
[[ 0.99998499 -0.00472433  0.0027747  -0.14402908]
 [ 0.00469899  0.99994783  0.00906924 -0.5465738 ]
 [-0.00281741 -0.00905606  0.99995502 -0.25793567]
 [ 0.          0.          0.          1.        ]]
{'MAE_accuracy (m)': 0.39565386227779686, 'MAE_completeness (m)': 0.4012337440875908, 'Chamfer_L1 (m)': 0.39844380318269385, 'Precision [Accuracy] (%)': 11.20384778610836, 'Recall [Completeness] (%)': 11.148073184906114, 'F-score (%)': 11.175890898509698, 'Spacing (m)': 0.05, 'Inlier_threshold (m)': 0.2, 'Outlier_truncation_acc (m)': 1, 'Outlier_truncation_com (m)': 1}
